# M65b A2 gradient diagnosis

Revision **M65B-ZERO-UPDATE-GRADIENTS-2026-10-05-r1**. Run sections **1–3 in order**, preferably on an A100 with CUDA 13.0. No phone or external teacher is needed.

M65 completed one preservation-control epoch, but Pedestrian 3D AP and Vehicle BEV AP regressed. This notebook measures the separate GT and preservation gradients on **32 fixed training images** at original A2 and the completed control checkpoint. It creates **no optimizer and takes zero parameter updates**. It does not retry training, tune weights, reconstruct historical gradients, or change the architecture.

Required on Drive: unchanged M65 manifest, original A2 epoch130 checkpoint, and completed M65 `control/checkpoint_epoch_1.pth` with its sidecar. A results ZIP alone does not contain the checkpoint tensors. Missing inputs stop before setup. Historical manifests and directories remain unchanged; a fresh private runtime and A2 source checkout receive separate diagnostic receipts.

Deliverable: `m65b_gradient_results.zip`. Stop for review after section 3.

## 1 Settings and project update

Check the M65 manifest path if your completed run used another folder. Keep its contents unchanged. Both checkpoint files must still exist. The revision printed below identifies the diagnostic notebook.

In [ ]:
# M65B-ZERO-UPDATE-GRADIENTS-2026-10-05-r1
from pathlib import Path
from collections import deque
import json, os, shlex, subprocess, sys, runpy, zipfile
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M65B-ZERO-UPDATE-GRADIENTS-2026-10-05-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
MANIFEST = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m65_a2_preservation/m65_a2_preservation_r1/m65_manifest.json')
if not MANIFEST.is_file(): raise FileNotFoundError(MANIFEST)
historical = json.loads(MANIFEST.read_text())
OUTPUT = Path(historical['output'])/'diagnostics_m65b'/'gradient_r1'
A2_REPO = Path('/content/MonoDETR_M65B_A2')
DATASET = Path('/content/kitti_m65b')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
PYTHON = Path('/content/m65b_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
for path in (Path(historical['models']['a2']['checkpoint']), Path(historical['output'])/'control/checkpoint_epoch_1.pth'):
    if not path.is_file(): raise FileNotFoundError(f'Preserve M65; missing diagnostic input: {path}')
OUTPUT.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
READY = False
print(REVISION, '\nHistorical M65 is read-only. Diagnostic output:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT/'colab_logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=60)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n')
        with subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            code = process.wait()
        log.write(f'\nEXIT CODE: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/probe_m65_preservation_gradients.py'
if not SCRIPT.is_file(): raise FileNotFoundError('The diagnostic is not published on main yet. Stop and reload after publication.')
sys.path.insert(0,str(MOBILE_REPO/'scripts'))
namespace = runpy.run_path(str(SCRIPT), run_name='m65b_revision_check')
if namespace['REVISION'] != REVISION: raise RuntimeError('Reload the latest notebook; script revision differs')


## 2 Fresh runtime source and training data

Reuse the tested CUDA13 setup and build helpers. No MonoPRIO checkout or teacher model is loaded. Runtime differences are recorded as a new prospective diagnostic, not hidden by editing old hashes.

In [ ]:
# Prospective runtime only: no historical manifest, source or build receipt is rewritten.
READY = False
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='m65b_runtime_helpers')
ENV = helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m64_runtime.py','--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',OUTPUT/'runtime_receipt.json'],'runtime',MOBILE_REPO)
url = 'https://github.com/ZrrSkywalker/MonoDETR.git'
commit = '6994b9f512400b258c6edb75f77423beb9c126f2'
if not A2_REPO.exists():
    run_logged(['git','clone','--no-checkout',url,A2_REPO],'clone_source')
    run_logged(['git','checkout','--detach',commit],'pin_source',A2_REPO)
if not (A2_REPO/'.git').is_dir(): raise RuntimeError('Preserve existing path and choose a fresh A2_REPO')
head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=A2_REPO,text=True).strip()
origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=A2_REPO,text=True).strip()
if head != commit or origin.rstrip('/') != url: raise RuntimeError('Unexpected source identity; do not reset it')
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'patch_sources',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py','--repo',A2_REPO,'--role','a2'],'build_attention',MOBILE_REPO)
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in ('/content/kitti_m65','/content/kitti_m64','/content/kitti_m62','/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti'):
    command += ['--candidate',candidate]
run_logged(command,'restore_data',MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_m65_regression_diagnostic.py','-v'],'cpu_tests',MOBILE_REPO)
READY = True
print('M65b setup ready. No optimizer or teacher was run.')


## 3 Measure gradients and return the results

Each checkpoint runs in a fresh process. Compare un-clipped gradient norms and directions for the complete losses and individual components. A negative cosine means locally opposing directions. Original-A2 preservation gradients should be near zero because the anchor is identical. The completed-control probe measures the endpoint, not all historical updates.

In [ ]:
# Backward measurements only. Never call the M65 training workflow here.
if not READY: raise RuntimeError('Run sections 1 and 2 in order first')
for role in ('original','control'):
    run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,
                '--manifest',MANIFEST,'--repo',A2_REPO,'--dataset-root',DATASET,
                '--role',role,'--report',OUTPUT/f'{role}_gradients.json'],
               role+'_gradients',MOBILE_REPO)
    report = json.loads((OUTPUT/f'{role}_gradients.json').read_text())
    assert report['complete'] and report['optimizer_steps'] == 0
    assert report['model_parameters_and_buffers_unchanged'] and report['anchor_unchanged']
    assert report['checkpoint_files_unchanged']
    print(role, 'GT vs preservation gradient measurements:')
    for row in report['rows']:
        print('batch',row['batch'],row['sample_ids'],row['gt_vs_preservation']['all'],flush=True)
bundle = OUTPUT/'m65b_gradient_results.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in (OUTPUT/'original_gradients.json',OUTPUT/'control_gradients.json',OUTPUT/'runtime_receipt.json'):
        archive.write(path,path.name)
    for path in sorted((OUTPUT/'colab_logs').glob('*.log')):
        archive.write(path,'colab_logs/'+path.name)
print('Return:', bundle)
print('STOP for review. No training, external-teacher KD, checkpoint selection or deployment is authorized.')
